# PyTorch Lightning con CNN sobre MNIST

Notebook de clase para introducir **PyTorch Lightning** a partir de una **CNN sencilla**.

## Objetivos

- Ver cómo se organiza un proyecto con `LightningModule`
- Entrenar una CNN sobre MNIST
- Usar validación, test, `EarlyStopping` y `ModelCheckpoint`
- Comparar mentalmente este enfoque con PyTorch "puro"

## 1. Instalación

Si trabajas en Google Colab o en un entorno sin Lightning instalado, ejecuta la siguiente celda.
Si ya lo tienes instalado, puedes saltarla.

In [ ]:
# Descomenta si hace falta instalar dependencias
# !pip install lightning torchvision

## 2. Imports

Aquí cargamos PyTorch, torchvision y Lightning.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchvision import datasets, transforms

import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

## 3. Comprobación del dispositivo

Lightning puede usar CPU o GPU automáticamente, pero viene bien saber qué hay disponible.

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo disponible:", device)

Dispositivo disponible: cpu


## 4. Definición del modelo

Vamos a crear una clase que hereda de `LightningModule`.

### Qué contiene esta clase

- La arquitectura de la red
- El `forward`
- El paso de entrenamiento
- El paso de validación
- El paso de test
- El optimizador y el scheduler

Esta es una de las ideas clave de Lightning: **organizar el entrenamiento en una única clase**.

In [4]:
class LitMNISTCNN(L.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        # CNN sencilla para imágenes 1x28x28
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Después de dos pooling: 28x28 -> 14x14 -> 7x7
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)      # (batch, 16, 28, 28)
        x = F.relu(x)
        x = self.pool(x)       # (batch, 16, 14, 14)

        x = self.conv2(x)      # (batch, 32, 14, 14)
        x = F.relu(x)
        x = self.pool(x)       # (batch, 32, 7, 7)

        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)        # (batch, 10)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        self.log("test_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("test_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=2
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1
            }
        }

## 5. Carga de datos

Usaremos **MNIST**, un dataset clásico de dígitos manuscritos.

### Transformación
Convertimos las imágenes a tensor en el rango `[0, 1]`.

In [ ]:
transform = transforms.ToTensor()

train_dataset_full = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    train_dataset_full, [train_size, val_size]
)

print("Tamaño train:", len(train_dataset))
print("Tamaño val:", len(val_dataset))
print("Tamaño test:", len(test_dataset))

## 6. DataLoaders

Los `DataLoader` agrupan las muestras en batches.

- `shuffle=True` en entrenamiento
- `shuffle=False` en validación y test

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

## 7. Creamos el modelo

Aquí instanciamos nuestra CNN.

In [ ]:
model = LitMNISTCNN(lr=1e-3)
model

## 8. Callbacks

Vamos a usar dos callbacks muy importantes:

### `EarlyStopping`
Detiene el entrenamiento si la métrica de validación deja de mejorar.

### `ModelCheckpoint`
Guarda automáticamente el mejor modelo.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=3,
    verbose=True
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="best-mnist-cnn-{epoch:02d}-{val_loss:.4f}"
)

logger = CSVLogger(save_dir="logs", name="mnist_cnn_lightning")

## 9. Trainer

`Trainer` coordina todo el proceso.

Aquí definimos:

- número máximo de épocas
- uso de CPU/GPU
- callbacks
- logger

In [ ]:
trainer = L.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices="auto",
    logger=logger,
    callbacks=[early_stopping, checkpoint_callback]
)

## 10. Entrenamiento

Ahora sí, entrenamos el modelo.

In [ ]:
trainer.fit(model, train_loader, val_loader)

## 11. Evaluación final

Probamos el mejor checkpoint sobre el conjunto de test.

In [ ]:
trainer.test(model, dataloaders=test_loader, ckpt_path="best")

## 12. Ruta del mejor modelo

Lightning ha guardado automáticamente el mejor checkpoint.

In [ ]:
print("Mejor checkpoint:")
print(checkpoint_callback.best_model_path)

## 13. Cargar el mejor modelo manualmente

Si quieres reutilizar el modelo más adelante, puedes cargarlo así:

In [ ]:
best_model = LitMNISTCNN.load_from_checkpoint(checkpoint_callback.best_model_path)
best_model.eval()
print("Modelo cargado correctamente")

## 14. Probar una predicción manual

Vamos a tomar una imagen del conjunto de test y ver qué predice la red.

In [ ]:
import matplotlib.pyplot as plt

x, y = test_dataset[0]
with torch.no_grad():
    logits = best_model(x.unsqueeze(0))
    pred = torch.argmax(logits, dim=1).item()

plt.imshow(x.squeeze(), cmap="gray")
plt.title(f"Etiqueta real: {y} | Predicción: {pred}")
plt.axis("off")
plt.show()

## 15. Qué aporta Lightning frente a PyTorch puro

Con este notebook ya tienes:

- modelo
- entrenamiento
- validación
- test
- métricas
- scheduler
- early stopping
- guardado del mejor modelo
- uso automático de GPU

Todo eso en PyTorch puro también se puede hacer, pero normalmente con más código repetitivo.

## 16. Mini-actividades para el alumnado

### Actividad 1
Cambia el optimizador `Adam` por `SGD`.

### Actividad 2
Añade una capa `Dropout` antes de `fc2`.

### Actividad 3
Modifica `EarlyStopping` para monitorizar `val_acc` en lugar de `val_loss`.

### Actividad 4
Cambia el tamaño de batch de 64 a 128 y compara resultados.

### Actividad 5
Añade una tercera capa convolucional y observa si mejora.

## 17. Preguntas de reflexión

1. ¿Qué partes del entrenamiento se escriben en Lightning y cuáles quedan ocultas?
2. ¿Qué ventajas tiene `EarlyStopping`?
3. ¿Por qué interesa guardar el mejor checkpoint y no solo el último?
4. ¿Qué diferencia hay entre train, validation y test?
5. ¿Cuándo usarías Lightning y cuándo PyTorch puro?